# ART-760 — Does the classifier beat chance?
## Monte Carlo simulation for empirical baseline estimation in imbalanced datasets

**How to cite:** *(add citation here)*

---

### Pipeline overview

| Cell | Module | Role |
|------|--------|------|
| 1 | module-00-setup | Packages, parameters, file resolution |
| 2 | module-01-synth-corpus | Synthetic corpus generation (bypass if Corpus exists) |
| 3 | module-02-read-validate | Corpus reading & validation |
| 4 | module-03-simulate | Monte Carlo simulation (bypass if Simulations exists) |
| 5 | *[OPTIONAL]* validation-known-cases | Known-outcome verification (run **instead of** module-03, then module-04) |
| 6 | module-04-report | Metric computation & reporting |
| 7 | module-05-stability | Iteration-sufficiency check (requires modules 00–04) |
| — | *[OPTIONAL]* module-05-save | Save provenance & stability table to Excel |
| — | *[OPTIONAL]* inspection | Per-iteration confusion matrix & metrics (run after module-04) |
| — | *[OPTIONAL]* inspection-save-excel | Save inspection table to Excel |
| — | *[OPTIONAL]* heterogeneous-classifiers | Heterogeneous reviewer simulation (replaces modules 01–03) |
| — | module-99-requirements | Dependency snapshot |

> **Normal execution:** run cells 1 → 6 in order, skipping the optional validation cell.  
> **Sufficiency check:** after module-04, run module-05-stability (optionally module-05-save).  
> **Validation run:** run cells 1 → 4 (module-03), then validation-known-cases **instead of** module-03, then module-04.

### module-00-setup

In [1]:
# module-00-setup
# Installs and imports all required packages, defines all global parameters,
# and resolves the path to the working Excel file.
# All downstream modules consume EXCEL_PATH without modification.

# ******************************************
# DEPENDENCIES
# ******************************************
import importlib, subprocess, sys

def ensure_packages(pkgs: dict):
    """Install missing packages, then import all. pkgs = {import_name: pip_name}"""
    for imp, pip in pkgs.items():
        if importlib.util.find_spec(imp) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip])
    for imp in pkgs:
        globals()[imp] = importlib.import_module(imp)

ensure_packages({
    "pandas":   "pandas",    # data wrangling
    "openpyxl": "openpyxl",  # read/write Excel files (.xlsx)
    "numpy":    "numpy",     # numerical operations & random sampling
})

import random
import numpy as np
import pandas as pd
from pathlib import Path
# ******************************************

# ******************************************
# PARAMETERS
# ******************************************
EXCEL_PATH = None                    # "C:/full/path/to/file.xlsx"  |  None → resolve below
EXCEL_FILE = "ART760_templateN-200_synthetic-B.xlsx"  # filename in notebook directory  |  None → file picker

POS_LABELS = ["included"]   # label(s) mapped to POSITIVE — use list for multiple
NEG_LABELS = ["excluded"]   # label(s) mapped to NEGATIVE — use list for multiple
N_ITER     = 10_000         # number of Monte Carlo iterations
SEED       = 34             # global random seed  |  None → no seed

SH_SPEC = "Spec"
SH_CORP = "Corpus"
SH_SIMS = "Simulations"
SH_REP  = "Reporting"
# ******************************************

# ******************************************
# Global seed
# ******************************************
if SEED is not None:
    random.seed(SEED)
    np.random.seed(SEED)
# ******************************************

# ******************************************
# Resolve Excel path
# ******************************************
def _resolve_excel(excel_path, filename, notebook_dir):
    """
    Priority:
      1. EXCEL_PATH set and file exists  → use directly.
      2. EXCEL_FILE set and found in notebook_dir  → build full path.
      3. Neither / not found  → open tkinter file picker.
    """
    # 1. Full path provided
    if excel_path is not None and str(excel_path).strip():
        p = Path(excel_path)
        if not p.exists():
            raise FileNotFoundError(f"EXCEL_PATH not found: {excel_path}")
        print(f"✔ Using EXCEL_PATH: {p}")
        return p

    # 2. Filename provided
    if filename is not None and str(filename).strip():
        p = notebook_dir / filename
        if p.exists():
            print(f"✔ File found: {p.name}")
            return p
        print(f"⚠ '{filename}' not found in {notebook_dir}. Opening file picker…")

    # 3. File picker (works in both Jupyter and VS Code Jupyter kernel)
    import tkinter as tk
    from tkinter import filedialog
    root = tk.Tk()
    root.withdraw()
    root.wm_attributes("-topmost", True)
    chosen = filedialog.askopenfilename(
        title="Select the scenario Excel file",
        initialdir=str(notebook_dir),
        filetypes=[("Excel files", "*.xlsx"), ("All files", "*.*")]
    )
    root.destroy()
    if not chosen:
        raise ValueError("No file selected.")
    p = Path(chosen)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {chosen}")
    print(f"✔ Selected: {p.name}")
    return p

# Resolve notebook directory: __file__ not available in notebooks,
# so fall back to Path.cwd() (which is the notebook directory in Jupyter).
_notebook_dir = Path.cwd()

EXCEL_PATH = _resolve_excel(EXCEL_PATH, EXCEL_FILE, _notebook_dir)
print(f"   Working directory: {_notebook_dir}")


✔ File found: ART760_templateN-200_synthetic-B.xlsx
   Working directory: m:\OneDrive - UPV\R\ART-760


### module-01-synth-corpus

In [7]:
# module-01-synth-corpus
# Generates a synthetic labelled corpus from the Spec sheet and writes it to
# the Corpus sheet. Automatically skipped if Corpus already contains data.
# Depends on: EXCEL_PATH, SH_CORP, SH_SPEC, SEED, POS_LABELS, NEG_LABELS (module-00)

# ******************************************
# Bypass guard — skip if Corpus already has data
# ******************************************
import openpyxl

def _read_sheet_safe(path, sheet_name):
    """Return DataFrame for sheet_name, or empty DataFrame if sheet missing/empty."""
    try:
        df = pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")
        return df
    except Exception:
        return pd.DataFrame()

_corpus_existing = _read_sheet_safe(EXCEL_PATH, SH_CORP)
_skip_m1 = len(_corpus_existing) > 0

if _skip_m1:
    print(f"⚠ Module 1 skipped: Corpus sheet already contains "
          f"{len(_corpus_existing)} rows. Proceed to Module 2.")
    print("  Label distribution in existing corpus:")
    print(_corpus_existing["Label"].value_counts().to_string())

del _corpus_existing
# ******************************************

if not _skip_m1:

    # ******************************************
    # Read Spec
    # ******************************************
    spec_raw = pd.read_excel(EXCEL_PATH, sheet_name=SH_SPEC, engine="openpyxl")

    required_spec = {"n_items", "Label", "Proportion"}
    missing_spec  = required_spec - set(spec_raw.columns)
    if missing_spec:
        raise ValueError(f"Spec sheet missing columns: {', '.join(missing_spec)}")

    spec_raw = spec_raw.dropna(subset=["Label", "Proportion"])

    n_items  = int(spec_raw["n_items"].iloc[0])
    levels_k = spec_raw["Label"].tolist()
    probs_raw = spec_raw["Proportion"].to_numpy(dtype=float)
    probs_k  = probs_raw / probs_raw.sum()   # renormalise

    if abs(probs_raw.sum() - 1.0) > 0.01:
        print("⚠ Spec proportions did not sum to 1 — renormalised automatically.")

    print(f"Spec loaded: {n_items} items | {len(levels_k)} labels: "
          + " | ".join(f"{l}={p:.3f}" for l, p in zip(levels_k, probs_k)))
    # ******************************************

    # ******************************************
    # Generate synthetic corpus
    # ******************************************
    rng = np.random.default_rng(SEED)

    counts = np.round(probs_k * n_items).astype(int)
    counts[0] = n_items - counts[1:].sum()   # adjust first to guarantee exact total

    label_vec = np.repeat(levels_k, counts)
    rng.shuffle(label_vec)

    corpus_synth = pd.DataFrame({
        "ID":       np.arange(1, n_items + 1),
        "Title":    [f"Synthetic reference {i}" for i in range(1, n_items + 1)],
        "Abstract": [f"Abstract placeholder for item {i}" for i in range(1, n_items + 1)],
        "Label":    label_vec,
    })

    print("Synthetic corpus generated:")
    print(corpus_synth["Label"].value_counts().to_string())
    # ******************************************

    # ******************************************
    # Write to Excel (_synthetic.xlsx)
    # ******************************************
    synth_path = EXCEL_PATH.parent / (EXCEL_PATH.stem + "_synthetic.xlsx")

    # Copy workbook then replace/add Corpus sheet
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Border, Side

    wb = load_workbook(EXCEL_PATH)
    if SH_CORP in wb.sheetnames:
        del wb[SH_CORP]
    ws = wb.create_sheet(SH_CORP)

    # Write header with style
    header_fill = PatternFill("solid", fgColor="E2EFDA")
    header_font = Font(bold=True)
    for col_idx, col_name in enumerate(corpus_synth.columns, start=1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.fill = header_fill
        cell.font = header_font

    # Write data rows
    for row_idx, row in enumerate(corpus_synth.itertuples(index=False), start=2):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=row_idx, column=col_idx, value=value)

    wb.save(synth_path)
    EXCEL_PATH = synth_path
    print(f"✔ Synthetic corpus written to '{EXCEL_PATH.name}'")
    # ******************************************


⚠ Module 1 skipped: Corpus sheet already contains 19 rows. Proceed to Module 2.
  Label distribution in existing corpus:
Label
included    10
excluded     9


### module-02-read-validate

In [8]:
# module-02-read-validate
# Reads and validates the Corpus and Spec sheets. Binarises labels into
# positive / negative. Outputs: corpus_df, spec_df, n_items, levels_k, probs_k.
# Depends on: EXCEL_PATH, SH_CORP, SH_SPEC, POS_LABELS, NEG_LABELS (module-00)

# ******************************************
# Read Corpus
# ******************************************
corpus_raw = pd.read_excel(EXCEL_PATH, sheet_name=SH_CORP, engine="openpyxl")

required_cols = {"ID", "Label"}
optional_cols = {"Title", "Abstract"}

missing_req = required_cols - set(corpus_raw.columns)
if missing_req:
    raise ValueError(f"Corpus sheet missing required columns: {', '.join(missing_req)}")

missing_opt = optional_cols - set(corpus_raw.columns)
if missing_opt:
    print(f"⚠ Corpus sheet missing optional columns: {', '.join(missing_opt)}")

print(f"Corpus loaded: {len(corpus_raw)} rows | {len(corpus_raw.columns)} columns")
# ******************************************

# ******************************************
# Binarise labels
# ******************************************
unknown_labels = set(corpus_raw["Label"].unique()) - set(POS_LABELS) - set(NEG_LABELS)
if unknown_labels:
    print(f"⚠ Unknown label(s) — will be set to NA in label_bin: "
          f"{', '.join(str(l) for l in unknown_labels)}")

def _binarise_series(series, pos, neg):
    """Map label series to 'positive' / 'negative' / NaN."""
    return series.map(
        lambda x: "positive" if x in pos else ("negative" if x in neg else None)
    )

corpus_df = corpus_raw.copy()
corpus_df["label_bin"] = _binarise_series(corpus_df["Label"], POS_LABELS, NEG_LABELS)

n_pos = (corpus_df["label_bin"] == "positive").sum()
n_neg = (corpus_df["label_bin"] == "negative").sum()
n_na  = corpus_df["label_bin"].isna().sum()

print(f"Label binarisation: {n_pos} positive | {n_neg} negative | {n_na} NA")
# ******************************************

# ******************************************
# Read Spec
# ******************************************
spec_df = pd.read_excel(EXCEL_PATH, sheet_name=SH_SPEC, engine="openpyxl")

required_spec = {"n_items", "Label", "Proportion"}
missing_spec  = required_spec - set(spec_df.columns)
if missing_spec:
    raise ValueError(f"Spec sheet missing columns: {', '.join(missing_spec)}")

spec_df  = spec_df.dropna(subset=["Label", "Proportion"])
n_items  = int(spec_df["n_items"].iloc[0])
levels_k = spec_df["Label"].tolist()
probs_raw = spec_df["Proportion"].to_numpy(dtype=float)
probs_k  = probs_raw / probs_raw.sum()

if abs(probs_raw.sum() - 1.0) > 0.01:
    print("⚠ Spec proportions did not sum to 1 — renormalised automatically.")

print(f"Spec loaded: {len(levels_k)} label levels | "
      + " | ".join(f"{l}={p:.3f}" for l, p in zip(levels_k, probs_k)))
# ******************************************

# ******************************************
# Corpus summary
# ******************************************
print("─── Corpus summary " + "─" * 30)
print(f"  Total items   : {len(corpus_df)}")
print(f"  Positive (n)  : {n_pos} ({n_pos / len(corpus_df) * 100:.1f}%)")
print(f"  Negative (n)  : {n_neg} ({n_neg / len(corpus_df) * 100:.1f}%)")
print(f"  NA label_bin  : {n_na}")
print(f"  Imbalance ratio (neg/pos): {n_neg / n_pos:.2f}" if n_pos > 0
      else "  Imbalance ratio: undefined (no positives)")
print("─" * 50)
# ******************************************


Corpus loaded: 19 rows | 4 columns
Label binarisation: 10 positive | 9 negative | 0 NA
Spec loaded: 2 label levels | included=0.526 | excluded=0.474
─── Corpus summary ──────────────────────────────
  Total items   : 19
  Positive (n)  : 10 (52.6%)
  Negative (n)  : 9 (47.4%)
  NA label_bin  : 0
  Imbalance ratio (neg/pos): 0.90
──────────────────────────────────────────────────


### module-03-simulate

In [9]:
# module-03-simulate
# Generates N_ITER simulated classification vectors and writes them to the
# Simulations sheet. Bypass: if the sheet already contains data, loads it as
# sims_df and skips simulation — Module 4 can then run directly.
# Depends on: corpus_df, n_items, levels_k, probs_k (module-02)
#             EXCEL_PATH, SH_SIMS, N_ITER, SEED (module-00)
# Exposes: _simulate_matrix() — reused by module-05-stability.

# ******************************************
# Simulation generator
# ******************************************
# Single source of truth for how simulated classifications are generated.
# Reused by module-05-stability. Do not duplicate this logic elsewhere.
# Defined ahead of the bypass guard so it stays available to Module 5 even
# when the simulation itself is skipped.
def _simulate_matrix(n_iter, seed, n_rows=None, lvls=None, prob=None):
    """
    Draw an (n_rows x n_iter) matrix of simulated labels in one vectorised call.
    Defaults resolve against the current corpus/Spec globals at call time.
    Returns a DataFrame with columns iter_0001 … iter_NNNN (no ID column).
    """
    if n_rows is None:
        n_rows = len(corpus_df)
    if lvls is None:
        lvls = levels_k
    if prob is None:
        prob = probs_k
    rng = np.random.default_rng(seed)
    m = rng.choice(lvls, size=(n_rows, n_iter), p=prob)
    return pd.DataFrame(
        m, columns=[f"iter_{i:04d}" for i in range(1, n_iter + 1)]
    )
# ******************************************

# ******************************************
# Bypass guard — skip if Simulations already has data
# ******************************************
_sims_existing = _read_sheet_safe(EXCEL_PATH, SH_SIMS)
_skip_m3 = len(_sims_existing) > 0

if _skip_m3:
    sims_df = _sims_existing
    print(f"⚠ Module 3 skipped: Simulations sheet already contains "
          f"{len(sims_df)} rows × {len(sims_df.columns) - 1} iterations. "
          f"Loaded as sims_df — proceed to Module 4.")
    del _sims_existing
# ******************************************

if not _skip_m3:
    del _sims_existing

    # ******************************************
    # Run simulation
    # ******************************************
    print(f"Running {N_ITER:,} iterations × {n_items:,} items…")

    sims_df = _simulate_matrix(n_iter=N_ITER, seed=SEED)
    sims_df.insert(0, "ID", corpus_df["ID"].values)

    print(f"✔ Simulation complete: {len(sims_df)} items × {N_ITER} iterations")
    # ******************************************

    # ******************************************
    # Write Simulations to Excel
    # ******************************************
    print("Writing to Excel…")

    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill

    wb = load_workbook(EXCEL_PATH)
    if SH_SIMS in wb.sheetnames:
        del wb[SH_SIMS]
    ws = wb.create_sheet(SH_SIMS)

    header_fill = PatternFill("solid", fgColor="FFF2CC")
    header_font = Font(bold=True)
    for col_idx, col_name in enumerate(sims_df.columns, start=1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.fill = header_fill
        cell.font = header_font

    for row_idx, row in enumerate(sims_df.itertuples(index=False), start=2):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=row_idx, column=col_idx, value=value)

    wb.save(EXCEL_PATH)
    print(f"✔ Simulations written to sheet '{SH_SIMS}' in {EXCEL_PATH.name}")
    # ******************************************


⚠ Module 3 skipped: Simulations sheet already contains 19 rows × 10000 iterations. Loaded as sims_df — proceed to Module 4.


### [OPTIONAL] validation-known-cases

> Run this cell **instead of module-03** if you want to verify that module-04
> computes metrics correctly. Skip it to run the normal Monte Carlo pipeline.
>
> Expected outcomes:
> | Vector | Recall | Specificity | Precision | MCC |
> |--------|--------|-------------|-----------|-----|
> | all_positive | 1 | 0 | prevalence | 0 |
> | all_negative | 0 | 1 | NA | 0 |
> | perfect | 1 | 1 | 1 | 1 |
> | random_fair | ≈0.5 | ≈0.5 | ≈prevalence | ≈0 |

In [12]:
# validation-known-cases  [OPTIONAL — run INSTEAD of module-03, then run module-04]
# Builds 4 known-outcome classification vectors, writes them to the
# Simulations sheet, and exposes them as sims_df for Module 4.
# Depends on: corpus_df (module-02) | EXCEL_PATH, SH_SIMS, POS_LABELS, NEG_LABELS, SEED (module-00)

# ******************************************
# Build validation vectors
# ******************************************
rng_val = np.random.default_rng(SEED)
n = len(corpus_df)

val_df = pd.DataFrame({
    "ID":           corpus_df["ID"].values,
    "all_positive": POS_LABELS[0],
    "all_negative": NEG_LABELS[0],
    "perfect":      corpus_df["Label"].values,
    "random_fair":  rng_val.choice(
                        [POS_LABELS[0], NEG_LABELS[0]],
                        size=n, p=[0.5, 0.5]
                    ),
})

print(f"Validation dataset built: {n} items × 4 cases")
print(f"  all_positive : all items classified as '{POS_LABELS[0]}'")
print(f"  all_negative : all items classified as '{NEG_LABELS[0]}'")
print(f"  perfect      : identical to gold standard")
print(f"  random_fair  : random 50/50 regardless of prevalence")
# ******************************************

# ******************************************
# Write to Excel (_validation.xlsx) & expose as sims_df
# ******************************************
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

val_path = EXCEL_PATH.parent / (EXCEL_PATH.stem + "_validation.xlsx")

wb = load_workbook(EXCEL_PATH)
if SH_SIMS in wb.sheetnames:
    del wb[SH_SIMS]
ws = wb.create_sheet(SH_SIMS)

header_fill = PatternFill("solid", fgColor="FFE0CC")
header_font = Font(bold=True)
for col_idx, col_name in enumerate(val_df.columns, start=1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

for row_idx, row in enumerate(val_df.itertuples(index=False), start=2):
    for col_idx, value in enumerate(row, start=1):
        ws.cell(row=row_idx, column=col_idx, value=value)

wb.save(val_path)
EXCEL_PATH = val_path

sims_df = val_df   # expose for Module 4

print(f"✔ Validation cases written to '{EXCEL_PATH.name}' — original preserved")
print("  → Now run module-04 to verify metrics")
# ******************************************

Validation dataset built: 130 items × 4 cases
  all_positive : all items classified as 'included'
  all_negative : all items classified as 'excluded'
  perfect      : identical to gold standard
  random_fair  : random 50/50 regardless of prevalence
✔ Validation cases written to 'ART760_templateBlank2_hetero_validation.xlsx' — original preserved
  → Now run module-04 to verify metrics


### module-04-report

In [10]:
# module-04-report
# Computes 8 classification metrics (Recall, Specificity, Precision, NPV,
# F1, F2, Balanced Accuracy, MCC) for each iteration in sims_df.
# Summarises empirical distributions and writes results to the Reporting sheet.
# Depends on: corpus_df, sims_df (module-02 / module-03 or validation cell)
#             EXCEL_PATH, SH_REP, POS_LABELS, NEG_LABELS (module-00)
# Exposes: _metrics_table(), _ci_lo(), _ci_hi(), metric_names — reused by
#          module-05-stability.

# ******************************************
# Helper functions
# ******************************************
def _binarise(series, pos, neg):
    """Map a label series to 'positive' / 'negative' / None."""
    return series.map(lambda x: "positive" if x in pos
                                else ("negative" if x in neg else None))

def _confusion(pred, truth):
    """Return TP, TN, FP, FN counts given two aligned Series."""
    tp = ((pred == "positive") & (truth == "positive")).sum()
    tn = ((pred == "negative") & (truth == "negative")).sum()
    fp = ((pred == "positive") & (truth == "negative")).sum()
    fn = ((pred == "negative") & (truth == "positive")).sum()
    return int(tp), int(tn), int(fp), int(fn)

def _metrics(tp, tn, fp, fn):
    """
    Compute 8 metrics from confusion matrix counts.
    Degenerate denominators (zero) → None (stored as NaN in DataFrame).
    MCC denominator uses float to avoid integer overflow on large corpora.
    """
    recall   = tp / (tp + fn)          if (tp + fn) > 0 else None
    spec     = tn / (tn + fp)          if (tn + fp) > 0 else None
    prec     = tp / (tp + fp)          if (tp + fp) > 0 else None
    npv      = tn / (tn + fn)          if (tn + fn) > 0 else None
    f1       = (2 * prec * recall / (prec + recall)
                if prec is not None and recall is not None
                   and (prec + recall) > 0 else None)
    f2       = (5 * prec * recall / (4 * prec + recall)
                if prec is not None and recall is not None
                   and (4 * prec + recall) > 0 else None)
    bal_acc  = ((recall + spec) / 2
                if recall is not None and spec is not None else None)
    mcc_den  = ((float(tp + fp) * float(tp + fn)
                 * float(tn + fp) * float(tn + fn)) ** 0.5)
    mcc      = (tp * tn - fp * fn) / mcc_den if mcc_den > 0 else None
    return {
        "Recall": recall, "Specificity": spec, "Precision": prec, "NPV": npv,
        "F1": f1, "F2": f2, "BalancedAccuracy": bal_acc, "MCC": mcc,
    }

def _metrics_table(pred_cols, truth_vec):
    """
    pred_cols: DataFrame where each column is one prediction vector.
    Single source of truth for assembling the per-iteration metrics table.
    Reused by module-05-stability.
    """
    rows = [
        _metrics(*_confusion(_binarise(pred_cols[c], POS_LABELS, NEG_LABELS),
                             truth_vec))
        for c in pred_cols.columns
    ]
    return pd.DataFrame(rows, index=list(pred_cols.columns))

# Single source of truth for the percentile-method confidence interval.
def _ci_lo(vals):
    return vals.dropna().quantile(0.025)

def _ci_hi(vals):
    return vals.dropna().quantile(0.975)
# ******************************************

# ******************************************
# Compute metrics per iteration
# ******************************************
truth     = corpus_df["label_bin"]
iter_cols = [c for c in sims_df.columns if c != "ID"]

print(f"Computing metrics for {len(iter_cols):,} iterations…")

metrics_mat = _metrics_table(sims_df[iter_cols], truth)
print(f"✔ Metrics computed: {len(metrics_mat)} iterations × {len(metrics_mat.columns)} metrics")
# ******************************************

# ******************************************
# Summarise empirical distributions
# ******************************************
metric_names = ["Recall", "Specificity", "Precision", "NPV",
                "F1", "F2", "BalancedAccuracy", "MCC"]

summary_rows = []
for m in metric_names:
    vals = metrics_mat[m].dropna()
    na_pct = round(metrics_mat[m].isna().mean() * 100, 1)
    summary_rows.append({
        "Metric":   m,
        "Mean":     vals.mean(),
        "Median":   vals.median(),
        "SD":       vals.std(),
        "CI95_Lo":  _ci_lo(vals),
        "CI95_Hi":  _ci_hi(vals),
        "NA_pct":   na_pct,
    })

report_df = pd.DataFrame(summary_rows)

print("─── Empirical baseline summary " + "─" * 19)
print(report_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("─" * 50)
# ******************************************

# ******************************************
# Write Reporting to Excel
# ******************************************
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, numbers

wb = load_workbook(EXCEL_PATH)
if SH_REP in wb.sheetnames:
    del wb[SH_REP]
ws = wb.create_sheet(SH_REP)

header_fill = PatternFill("solid", fgColor="DCE6F1")
header_font = Font(bold=True)
for col_idx, col_name in enumerate(report_df.columns, start=1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

num_fmt = "0.0000"
for row_idx, row in enumerate(report_df.itertuples(index=False), start=2):
    for col_idx, value in enumerate(row, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=value)
        if col_idx > 1 and col_idx < len(report_df.columns):  # numeric cols, not NA_pct
            cell.number_format = num_fmt

wb.save(EXCEL_PATH)
print(f"✔ Report written to sheet '{SH_REP}' in {EXCEL_PATH.name}")
# ******************************************


Computing metrics for 10,000 iterations…
✔ Metrics computed: 10000 iterations × 8 metrics
─── Empirical baseline summary ───────────────────
          Metric   Mean  Median     SD  CI95_Lo  CI95_Hi  NA_pct
          Recall 0.5267  0.5000 0.1586   0.2000   0.8000  0.0000
     Specificity 0.4753  0.4444 0.1663   0.1111   0.7778  0.0000
       Precision 0.5272  0.5333 0.1192   0.2857   0.7505  0.0000
             NPV 0.4752  0.5000 0.1331   0.2217   0.7500  0.0000
              F1 0.5208  0.5263 0.1262   0.2353   0.7500  0.1000
              F2 0.5231  0.5102 0.1431   0.2128   0.7843  0.1000
BalancedAccuracy 0.5010  0.5111 0.1154   0.2667   0.7333  0.0000
             MCC 0.0022  0.0272 0.2372  -0.4778   0.4778  0.0000
──────────────────────────────────────────────────
✔ Report written to sheet 'Reporting' in ART-760_alfalla19.xlsx


### module-05-stability

The default of 10,000 iterations is a convention, not a derivation. This module
verifies empirically that it is sufficient for the corpus at hand. The user
declares a target precision on the metric scale before running the check — the
default of 0.01 follows from the way the bounds are used: they are reported and
interpreted to two decimal places, so variation below 0.01 cannot alter any
interpretation of the results. The full simulation is replicated
`N_REPLICATIONS` times with different random seeds and all other parameters
unchanged, and for each of the eight metrics the module reports the largest
variation observed in **both** ends of the CI95 between replications. Where the
target is not met, `N_Required` estimates the iteration count needed, derived
from the 1/√N scaling of percentile sampling error.

Requires modules 00 → 04 in the current session: it reuses `_simulate_matrix()`,
`_metrics_table()`, `_ci_lo()` and `_ci_hi()` from the global namespace and does
not read the Excel file. Replication 1 reuses `SEED`, so its bounds are compared
against `report_df` as an automatic check that the replication engine still
reproduces the main pipeline exactly.

> **Runtime** scales as `N_REPLICATIONS × N_ITER_CHECK` — roughly ten times a
> single module-04 run at the default setting. Because scenarios live in
> separate Excel files, run this module once per scenario.

In [11]:
# module-05-stability
# Checks whether the iteration count is large enough by replicating the full
# simulation with different seeds and measuring how far BOTH ends of the CI95
# move between replications. Also estimates the iteration count that would be
# needed to reach the declared target precision.
#
# REQUIRES modules 00-04 in the current session. Reuses _simulate_matrix()
# from module-03 and _metrics_table(), _ci_lo(), _ci_hi(), metric_names from
# module-04. Does not read the Excel file and does not touch sims_df/report_df.

# ******************************************
# PARAMETERS
# ******************************************
N_ITER_CHECK     = N_ITER   # iterations per replication. Defaults to the main
                            # experiment. Override with a literal (e.g. 80_000)
                            # to explore what-if scenarios; when it differs from
                            # N_ITER the consistency check against report_df is
                            # skipped, since replication 1 no longer reproduces
                            # the stored run.

N_REPLICATIONS   = 10       # independent replications, each of N_ITER_CHECK
                            # iterations with a different seed. This is NOT the
                            # iteration count: N_ITER_CHECK defines the empirical
                            # distribution, N_REPLICATIONS measures how
                            # reproducible its bounds are. Cost grows as
                            # N_REPLICATIONS x N_ITER_CHECK. Keep it fixed if you
                            # compare Range values across runs: the sufficiency
                            # estimate below assumes it unchanged.

TARGET_PRECISION = 0.01     # maximum acceptable spread of a bound across
                            # replications, on the metric scale. 0.01 = bounds
                            # are read to two decimals, so anything below this
                            # cannot change the interpretation.

BLOCK_SIZE       = 2000     # prediction columns processed per metrics block.
                            # Bounds peak memory when N_ITER_CHECK is large;
                            # cannot affect the result.
# ******************************************

# ******************************************
# Dependency check
# ******************************************
import math, sys, time
from datetime import datetime

_needed = ["corpus_df", "levels_k", "probs_k", "N_ITER", "SEED",
           "metric_names", "_simulate_matrix", "_metrics_table",
           "_ci_lo", "_ci_hi"]
_absent = [nm for nm in _needed if nm not in globals()]
if _absent:
    raise RuntimeError("Module 5 requires modules 00-04 to be run first. "
                       "Missing: " + ", ".join(_absent))
del _needed, _absent
# ******************************************

# ******************************************
# Replication engine
# ******************************************
# No simulation or metric logic is restated here: generation comes from
# _simulate_matrix() (module-03), the metrics table from _metrics_table() and
# the bounds from _ci_lo()/_ci_hi() (module-04).
bound_keys = [f"{m}|{b}" for m in metric_names for b in ("CI95_Lo", "CI95_Hi")]

def _bounds_one_run(seed, n_iter):
    """Return the 16 CI95 bounds of one run, Lo/Hi paired within each metric."""
    sim = _simulate_matrix(n_iter=n_iter, seed=seed)

    # Metrics are computed in blocks of columns so that peak memory stays
    # bounded when n_iter is large. Blocking cannot affect the result: the whole
    # matrix is generated in a single call before any metric is computed.
    blocks = [sim.iloc[:, i:i + BLOCK_SIZE]
              for i in range(0, n_iter, BLOCK_SIZE)]
    m = pd.concat([_metrics_table(b, corpus_df["label_bin"]) for b in blocks])

    out = {}
    for nm in metric_names:
        out[f"{nm}|CI95_Lo"] = _ci_lo(m[nm])
        out[f"{nm}|CI95_Hi"] = _ci_hi(m[nm])
    return out
# ******************************************

# ******************************************
# Run replications
# ******************************************
# Replication 1 reuses SEED so it reproduces the run reported by module-04.
# The remaining seeds are drawn from a stream initialised with SEED itself:
# widely dispersed rather than consecutive, and fully recoverable from SEED
# alone, so the whole stability check stays reproducible.
_rng_seeds  = np.random.default_rng(SEED)
seed_vector = [SEED] + _rng_seeds.integers(
    1, 2**31 - 1, size=N_REPLICATIONS - 1).tolist()

print(f"Replicating the full simulation {N_REPLICATIONS} times "
      f"({N_ITER_CHECK:,} iterations x {len(corpus_df):,} items each)...")

_cols = {}
for _r, _sd in enumerate(seed_vector, start=1):
    _t0 = time.time()
    _cols[f"rep_{_r:02d}"] = _bounds_one_run(_sd, N_ITER_CHECK)
    print(f"  replication {_r}/{N_REPLICATIONS}  seed = {_sd}  "
          f"({time.time() - _t0:.1f}s)")

bounds_mat = pd.DataFrame(_cols).reindex(bound_keys)
del _cols, _r, _sd, _t0
# ******************************************

# ******************************************
# Consistency check against report_df
# ******************************************
_comparable = ("sims_df" in globals() and "report_df" in globals()
               and N_ITER_CHECK == N_ITER
               and sum(c.startswith("iter_") for c in sims_df.columns) == N_ITER)

if _comparable:
    _ref = pd.Series(
        [v for row in report_df.itertuples()
           for v in (row.CI95_Lo, row.CI95_Hi)],
        index=[f"{row.Metric}|{b}" for row in report_df.itertuples()
                                   for b in ("CI95_Lo", "CI95_Hi")]
    ).reindex(bound_keys)
    _delta = (bounds_mat["rep_01"] - _ref).abs().max()
    if _delta < 1e-12:
        print("Consistency check passed: replication 1 reproduces both bounds "
              "of report_df exactly.")
    else:
        print(f"WARNING: replication 1 does not reproduce report_df "
              f"(max abs diff = {_delta:.3g}). "
              f"Reconcile before reporting these results.")
    del _ref, _delta
elif N_ITER_CHECK != N_ITER:
    print(f"Consistency check skipped: N_ITER_CHECK ({N_ITER_CHECK:,}) differs "
          f"from N_ITER ({N_ITER:,}) - what-if mode.")
else:
    print("Consistency check skipped: sims_df was not produced by module-03.")
del _comparable
# ******************************************

# ******************************************
# Stability table
# ******************************************
# Required N follows from the 1/sqrt(N) scaling of percentile sampling error:
#   N_required = N_current * (Range_observed / Range_target)^2
# It is an approximation: Range is estimated from N_REPLICATIONS values only,
# and it assumes N_REPLICATIONS is held constant.
_rep_cols = list(bounds_mat.columns)

stability_df = bounds_mat.reset_index(drop=True)
stability_df.insert(0, "Metric", [k.split("|")[0] for k in bound_keys])
stability_df.insert(1, "Bound",  [k.split("|")[1] for k in bound_keys])

stability_df["Bound_Min"] = bounds_mat[_rep_cols].min(axis=1).values
stability_df["Bound_Max"] = bounds_mat[_rep_cols].max(axis=1).values
stability_df["Range"]     = stability_df["Bound_Max"] - stability_df["Bound_Min"]
stability_df["Target"]    = TARGET_PRECISION

# A bound that is NA in every replication yields Range = NA and cannot be
# judged; it is carried as pd.NA and excluded from the verdict, not counted
# as a failure.
stability_df["Meets_Target"] = (
    (stability_df["Range"] < TARGET_PRECISION)
    .where(stability_df["Range"].notna())
    .astype("boolean")
)

_met    = stability_df["Meets_Target"].fillna(True).to_numpy(dtype=bool)
_rng_np = stability_df["Range"].to_numpy(dtype=float)
_gradable = ~np.isnan(_rng_np) & (_rng_np != 0)
stability_df["N_Required"] = np.where(
    _met | ~_gradable,
    np.nan,
    np.ceil(N_ITER_CHECK * (_rng_np / TARGET_PRECISION) ** 2)
)
del _rep_cols, _rng_np, _gradable

print(f"--- CI95 bound stability across {N_REPLICATIONS} replications "
      f"({N_ITER_CHECK:,} iterations each) ---")
print(stability_df.round(4).to_string(index=False))
# ******************************************

# ******************************************
# Verdict and recommendation
# ******************************************
def _round_up(x):
    """Round up to two significant figures - used for the suggested N."""
    k = 10 ** (math.floor(math.log10(x)) - 1)
    return int(math.ceil(x / k) * k)

_w = int(stability_df["Range"].idxmax())   # idxmax skips NA

print("")
print(f"  Target precision       : {TARGET_PRECISION}")
print(f"  Largest range observed : {stability_df['Range'][_w]:.3g}"
      f"  ({stability_df['Metric'][_w]} {stability_df['Bound'][_w]})")

_exact = (stability_df["Range"] == 0) & stability_df["Range"].notna()
if _exact.any():
    print("  Identical in all replications: "
          + ", ".join(f"{m} {b}" for m, b in
                      zip(stability_df.loc[_exact, "Metric"],
                          stability_df.loc[_exact, "Bound"])))

if stability_df["Meets_Target"].all(skipna=True):
    print(f"  {N_ITER_CHECK:,} iterations are sufficient at the declared "
          f"target precision.")
    N_SUGGESTED = N_ITER_CHECK
else:
    _fail    = stability_df[~stability_df["Meets_Target"].fillna(True)]
    _max_req = _fail["N_Required"].max()
    _drv     = _fail.loc[_fail["N_Required"].idxmax()]
    N_SUGGESTED = _round_up(1.2 * _max_req)
    print("  Target not met for: "
          + ", ".join(f"{m} {b}" for m, b in
                      zip(_fail["Metric"], _fail["Bound"])))
    print(f"  Estimated requirement: {int(_max_req):,} iterations "
          f"(driven by {_drv['Metric']} {_drv['Bound']})")
    print(f"  Suggested value, 20% margin: N_ITER_CHECK = {N_SUGGESTED:,}")
    print("  Re-run this cell with that value to confirm "
          "before changing N_ITER.")
    del _fail, _max_req, _drv
del _exact, _met
# ******************************************

# ******************************************
# Provenance block
# ******************************************
# Everything needed to reproduce this check from SEED alone.
_lab = corpus_df["label_bin"].dropna()

stability_meta = pd.DataFrame({
    "Item": [
        "Corpus items", "Positive items", "Prevalence",
        "Iterations per replication (N_ITER_CHECK)",
        "Iterations in main experiment (N_ITER)",
        "Replications (N_REPLICATIONS)", "Target precision",
        "Base seed (SEED)", "Seed derivation", "Replication seeds",
        "Largest range observed", "Bound with largest range",
        "All bounds meet target", "Suggested iterations",
        "Python version", "Run timestamp",
    ],
    "Value": [
        str(len(corpus_df)),
        str(int(_lab.eq("positive").sum())),
        f"{_lab.eq('positive').mean():.4f}",
        str(N_ITER_CHECK), str(N_ITER),
        str(N_REPLICATIONS), str(TARGET_PRECISION), str(SEED),
        "rng = np.random.default_rng(SEED); "
        "[SEED] + rng.integers(1, 2**31 - 1, size=N_REPLICATIONS - 1)",
        ", ".join(str(s) for s in seed_vector),
        f"{stability_df['Range'][_w]:.3g}",
        f"{stability_df['Metric'][_w]} {stability_df['Bound'][_w]}",
        "yes" if stability_df["Meets_Target"].all(skipna=True) else "no",
        f"{N_SUGGESTED:,}",
        sys.version.split()[0],
        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    ],
})

print("--- Provenance ---")
print(stability_meta.to_string(index=False))
del _w, _lab
# ******************************************


Replicating the full simulation 10 times (10,000 iterations x 19 items each)...
  replication 1/10  seed = 34  (9.0s)
  replication 2/10  seed = 135177642  (8.2s)
  replication 3/10  seed = 8650587  (8.3s)
  replication 4/10  seed = 253779630  (9.5s)
  replication 5/10  seed = 1872985660  (8.8s)
  replication 6/10  seed = 244631417  (8.5s)
  replication 7/10  seed = 521285308  (8.0s)
  replication 8/10  seed = 15575661  (8.2s)
  replication 9/10  seed = 1398143490  (7.9s)
  replication 10/10  seed = 211968524  (8.1s)
--- CI95 bound stability across 10 replications (10,000 iterations each) ---
          Metric   Bound  rep_01  rep_02  rep_03  rep_04  rep_05  rep_06  rep_07  rep_08  rep_09  rep_10  Bound_Min  Bound_Max  Range  Target  Meets_Target  N_Required
          Recall CI95_Lo  0.2000  0.2000  0.2000  0.2000  0.2000  0.2000  0.2000  0.2000  0.2000  0.2000     0.2000     0.2000 0.0000    0.01          True         NaN
          Recall CI95_Hi  0.8000  0.8000  0.8000  0.8000  0.8000

---
### [OPTIONAL] module-05-save

> Writes the provenance block and the stability table to a new `Stability`
> sheet. Run only if you want them persisted — the sheet documents that the
> iteration count is large enough for this corpus and does not affect any
> result reported by modules 01–04.
>
> The provenance block sits at the top; the stability table follows after one
> blank row.

In [12]:
# module-05-save  [OPTIONAL — run after module-05-stability]
# Writes the provenance block and stability_df to the Stability sheet.
# Depends on: stability_meta, stability_df, bounds_mat (module-05-stability)
#             EXCEL_PATH (module-00)

# ******************************************
# Write Stability to Excel
# ******************************************
SH_STAB = "Stability"

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

wb = load_workbook(EXCEL_PATH)
if SH_STAB in wb.sheetnames:
    del wb[SH_STAB]
ws = wb.create_sheet(SH_STAB)

header_fill = PatternFill("solid", fgColor="D9EAD3")
header_font = Font(bold=True)

def _xl(value):
    """Coerce numpy/pandas scalars to native types openpyxl can write."""
    if pd.isna(value):
        return None
    return value.item() if hasattr(value, "item") else value

def _write_block(ws, df, start_row):
    """Write df with a styled header row at start_row. Returns last row used."""
    for col_idx, col_name in enumerate(df.columns, start=1):
        cell = ws.cell(row=start_row, column=col_idx, value=col_name)
        cell.fill = header_fill
        cell.font = header_font
    for row_offset, row in enumerate(df.itertuples(index=False), start=1):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=start_row + row_offset, column=col_idx,
                    value=_xl(value))
    return start_row + len(df)

# Provenance first, stability table below it with one blank row in between.
_write_block(ws, stability_meta, start_row=1)
start_stab = len(stability_meta) + 3
_write_block(ws, stability_df, start_row=start_stab)

# ******************************************
# Number formats on the stability table body
# ******************************************
dec_cols = set(bounds_mat.columns) | {"Bound_Min", "Bound_Max", "Range", "Target"}
int_cols = {"N_Required"}
body_rows = range(start_stab + 1, start_stab + 1 + len(stability_df))

for col_idx, col_name in enumerate(stability_df.columns, start=1):
    if col_name in dec_cols:
        fmt = "0.0000"
    elif col_name in int_cols:
        fmt = "#,##0"
    else:
        continue
    for row_idx in body_rows:
        ws.cell(row=row_idx, column=col_idx).number_format = fmt
# ******************************************

wb.save(EXCEL_PATH)
print(f"✔ Provenance and stability table written to sheet '{SH_STAB}' "
      f"in {EXCEL_PATH.name}")
# ******************************************


✔ Provenance and stability table written to sheet 'Stability' in ART-760_alfalla19.xlsx


---
### [OPTIONAL] inspection

> Displays the confusion matrix (TP, TN, FP, FN) and all 8 metrics for each
> iteration. Runs on `metrics_mat` and `sims_df` produced by the most recent
> module-04 execution.


In [13]:
# inspection  [OPTIONAL — run after module-04]
# Displays per-iteration confusion matrix and 8 metrics.
# Depends on: corpus_df, sims_df, metrics_mat (module-04)
#             POS_LABELS, NEG_LABELS (module-00)

# ******************************************
# Build per-iteration inspection table
# ******************************************
truth_insp = corpus_df["label_bin"]
iter_cols_insp = [c for c in sims_df.columns if c != "ID"]

insp_rows = []
for col in iter_cols_insp:
    pred = _binarise(sims_df[col], POS_LABELS, NEG_LABELS)
    tp, tn, fp, fn = _confusion(pred, truth_insp)
    row = {"Classifier": col, "TP": tp, "TN": tn, "FP": fp, "FN": fn}
    row.update(metrics_mat.loc[col].to_dict())
    insp_rows.append(row)

inspection_df = pd.DataFrame(insp_rows)
# ******************************************

# ******************************************
# Display
# ******************************************
print(f"─── Per-iteration inspection ({len(inspection_df)} rows) " + "─" * 10)
_display_cols = ["Classifier", "TP", "TN", "FP", "FN"] + [
    "Recall", "Specificity", "Precision", "NPV",
    "F1", "F2", "BalancedAccuracy", "MCC"
]
print(
    inspection_df[_display_cols]
    .to_string(index=False, float_format=lambda x: f"{x:.4f}")
)
print("─" * 50)
# ******************************************


─── Per-iteration inspection (10000 rows) ──────────
Classifier  TP  TN  FP  FN  Recall  Specificity  Precision    NPV     F1     F2  BalancedAccuracy     MCC
 iter_0001   7   5   4   3  0.7000       0.5556     0.6364 0.6250 0.6667 0.6863            0.6278  0.2584
 iter_0002   6   6   3   4  0.6000       0.6667     0.6667 0.6000 0.6316 0.6122            0.6333  0.2667
 iter_0003   6   5   4   4  0.6000       0.5556     0.6000 0.5556 0.6000 0.6000            0.5778  0.1556
 iter_0004   6   5   4   4  0.6000       0.5556     0.6000 0.5556 0.6000 0.6000            0.5778  0.1556
 iter_0005   4   2   7   6  0.4000       0.2222     0.3636 0.2500 0.3810 0.3922            0.3111 -0.3820
 iter_0006   5   5   4   5  0.5000       0.5556     0.5556 0.5000 0.5263 0.5102            0.5278  0.0556
 iter_0007   4   3   6   6  0.4000       0.3333     0.4000 0.3333 0.4000 0.4000            0.3667 -0.2667
 iter_0008   5   6   3   5  0.5000       0.6667     0.6250 0.5455 0.5556 0.5208            0.5833  

---
### [OPTIONAL] inspection-save-excel

> Saves the inspection table produced by the **inspection** cell to an
> `Inspection` sheet in the Excel file. Run only if you want to persist the
> per-iteration detail.


In [14]:
# inspection-save-excel  [OPTIONAL — run after inspection cell]
# Writes inspection_df to the Inspection sheet in the Excel file.
# Depends on: inspection_df (inspection cell) | EXCEL_PATH (module-00)

# ******************************************
# Write Inspection to Excel
# ******************************************
SH_INS = "Inspection"

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

wb = load_workbook(EXCEL_PATH)
if SH_INS in wb.sheetnames:
    del wb[SH_INS]
ws = wb.create_sheet(SH_INS)

header_fill = PatternFill("solid", fgColor="EAD1DC")
header_font = Font(bold=True)
for col_idx, col_name in enumerate(inspection_df.columns, start=1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

num_fmt = "0.0000"
metric_cols_idx = {
    col: idx for idx, col in enumerate(inspection_df.columns, start=1)
    if col in {"Recall", "Specificity", "Precision", "NPV",
               "F1", "F2", "BalancedAccuracy", "MCC"}
}
for row_idx, row in enumerate(inspection_df.itertuples(index=False), start=2):
    for col_idx, value in enumerate(row, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=value)
        if col_idx in metric_cols_idx.values():
            cell.number_format = num_fmt

wb.save(EXCEL_PATH)
print(f"✔ Inspection table written to sheet '{SH_INS}' in {EXCEL_PATH.name}")
# ******************************************


✔ Inspection table written to sheet 'Inspection' in ART-760_alfalla19.xlsx


---
### [OPTIONAL] heterogeneous-classifiers

> Generates a synthetic dataset of heterogeneous classifiers simulating human
> reviewers with different labelling biases. Replaces the Simulations sheet so
> **module-04** can be run directly after this cell (modules 01–03 will auto-bypass).
>
> Results are saved to a new file with suffix `_hetero.xlsx` to preserve the original.


In [15]:
# heterogeneous-classifiers  [OPTIONAL]
# Generates N_CLASSIFIERS synthetic classifiers with heterogeneous labelling
# biases. Saves to [original]_hetero.xlsx and updates EXCEL_PATH.
# After this cell: run module-04 directly.
# Depends on: EXCEL_PATH, POS_LABELS, NEG_LABELS, SEED (module-00)

# ******************************************
# PARAMETERS
# ******************************************
N_CLASSIFIERS = 20

PROFILE = "fixed"
# "fixed"  — cycle through 6 predefined bias profiles
#            (lenient, strict, random, prevalence, biased_pos, biased_neg)
# "random" — each classifier gets a p_pos drawn uniformly from P_POS_RANGE

P_POS_RANGE = (0.05, 0.95)
# Only used when PROFILE == "random".
# Range from which each classifier's p_pos is drawn.
# ******************************************

# ******************************************
# Corpus — load or generate from Spec
# ******************************************
# Sanity-check: if corpus_df is already in the namespace, verify it matches Spec.
_spec_check = _read_sheet_safe(EXCEL_PATH, SH_SPEC)
if "corpus_df" in dir() and len(_spec_check) > 0:
    _spec_check = _spec_check.dropna(subset=["Label", "Proportion"])
    _expected_n = int(_spec_check["n_items"].iloc[0])
    if len(corpus_df) != _expected_n:
        print(f"⚠ corpus_df has {len(corpus_df)} rows but Spec expects "
              f"{_expected_n}. Reloading from Corpus sheet.")
        del corpus_df
    else:
        print(f"✔ corpus_df matches Spec: {len(corpus_df)} rows — reusing.")

_corpus_raw_h = _read_sheet_safe(EXCEL_PATH, SH_CORP)

if len(_corpus_raw_h) > 0:
    print(f"Corpus sheet found: {len(_corpus_raw_h)} rows — loaded as-is.")
    corpus_df = _corpus_raw_h.copy()
    corpus_df["label_bin"] = _binarise_series(
        corpus_df["Label"], POS_LABELS, NEG_LABELS
    )
else:
    print("Corpus sheet empty — generating from Spec…")
    _spec_h = pd.read_excel(EXCEL_PATH, sheet_name=SH_SPEC, engine="openpyxl")
    _spec_h  = _spec_h.dropna(subset=["Label", "Proportion"])
    _n       = int(_spec_h["n_items"].iloc[0])
    _lvls    = _spec_h["Label"].tolist()
    _probs   = _spec_h["Proportion"].to_numpy(dtype=float)
    _probs  /= _probs.sum()

    rng_h = np.random.default_rng(SEED)
    _counts    = np.round(_probs * _n).astype(int)
    _counts[0] = _n - _counts[1:].sum()
    _lvec = np.repeat(_lvls, _counts)
    rng_h.shuffle(_lvec)

    corpus_df = pd.DataFrame({
        "ID":       np.arange(1, _n + 1),
        "Title":    [f"Synthetic reference {i}" for i in range(1, _n + 1)],
        "Abstract": [f"Abstract placeholder for item {i}" for i in range(1, _n + 1)],
        "Label":    _lvec,
    })
    corpus_df["label_bin"] = _binarise_series(
        corpus_df["Label"], POS_LABELS, NEG_LABELS
    )

n_items_h  = len(corpus_df)
prevalence = (corpus_df["label_bin"] == "positive").mean()
print(f"Corpus ready: {n_items_h} items | prevalence = {prevalence:.3f}")
del _corpus_raw_h
# ******************************************

# ******************************************
# Classifier profiles
# ******************************************
_fixed_profiles = pd.DataFrame({
    "profile": ["lenient", "strict", "random",
                "prevalence", "biased_pos", "biased_neg"],
    "p_pos":   [0.75, 0.25, 0.50, prevalence, 0.90, 0.10],
})

rng_cls = np.random.default_rng(SEED)

if PROFILE == "fixed":
    _idx      = [(i % len(_fixed_profiles)) for i in range(N_CLASSIFIERS)]
    _sel      = _fixed_profiles.iloc[_idx].reset_index(drop=True)
    p_pos_vec = _sel["p_pos"].tolist()
    label_vec_cls = _sel["profile"].tolist()
else:
    p_pos_vec     = rng_cls.uniform(P_POS_RANGE[0], P_POS_RANGE[1],
                                    size=N_CLASSIFIERS).tolist()
    label_vec_cls = [f"random_{i:02d}" for i in range(1, N_CLASSIFIERS + 1)]

col_names_cls = [f"{lbl}_{i:02d}"
                 for i, lbl in enumerate(label_vec_cls, start=1)]

print("Classifier profiles assigned:")
for nm, p in zip(col_names_cls, p_pos_vec):
    print(f"  {nm:<25}  p_pos={p:.3f}")
# ******************************************

# ******************************************
# Generate classifications
# ******************************************
def _classify_one(p_pos, n, rng):
    return rng.choice(
        [POS_LABELS[0], NEG_LABELS[0]],
        size=n, p=[p_pos, 1 - p_pos]
    )

sims_dict = {"ID": corpus_df["ID"].values}
for nm, p in zip(col_names_cls, p_pos_vec):
    sims_dict[nm] = _classify_one(p, n_items_h, rng_cls)

sims_df = pd.DataFrame(sims_dict)

print(f"✔ {N_CLASSIFIERS} classifiers generated: "
      f"{len(sims_df)} items × {len(sims_df.columns) - 1} classifiers")
# ******************************************

# ******************************************
# Write to Excel (_hetero.xlsx)
# ******************************************
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill

hetero_path = EXCEL_PATH.parent / (
    EXCEL_PATH.stem.replace("_synthetic", "") + "_hetero.xlsx"
)

wb = load_workbook(EXCEL_PATH)
if SH_SIMS in wb.sheetnames:
    del wb[SH_SIMS]
ws = wb.create_sheet(SH_SIMS)

header_fill = PatternFill("solid", fgColor="FFF2CC")
header_font = Font(bold=True)
for col_idx, col_name in enumerate(sims_df.columns, start=1):
    cell = ws.cell(row=1, column=col_idx, value=col_name)
    cell.fill = header_fill
    cell.font = header_font

for row_idx, row in enumerate(sims_df.itertuples(index=False), start=2):
    for col_idx, value in enumerate(row, start=1):
        ws.cell(row=row_idx, column=col_idx, value=value)

wb.save(hetero_path)
EXCEL_PATH = hetero_path
print(f"✔ Simulations written to '{EXCEL_PATH.name}' "
      f"— original preserved — proceed to module-04")
# ******************************************


IndexError: single positional indexer is out-of-bounds

### module-99-requirements

In [ ]:
# module-99-requirements
# Writes a requirements.txt snapshot of the current environment.
# Run at the end of any session to document dependency versions.

import subprocess, pathlib

req_path = pathlib.Path.cwd() / "requirements.txt"
with open(req_path, "w") as fh:
    subprocess.run(["pip", "freeze"], stdout=fh, check=True)

print(f"✔ requirements.txt written to {req_path}")
